# Attention Networks

Demonstrates an encoder-decoder LSTM with Bahdanau attention for sequence memorisation.

[View in Colaboratory](https://colab.research.google.com/github/stabgan/FastText-Sentence2Vec/blob/master/Attention_Networks.ipynb)

In [0]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, RepeatVector, TimeDistributed
from tensorflow.keras import backend as K
from tensorflow.keras import regularizers, constraints, initializers, activations
from tensorflow.keras.layers import Layer, InputSpec

from random import randint
from numpy import array, argmax, array_equal

In [0]:
def _time_distributed_dense(x, w, b, input_dim, timesteps, output_dim):
    """Apply a dense layer to every temporal slice of an input tensor.

    Replaces the removed keras.layers.recurrent._time_distributed_dense.
    """
    x = tf.reshape(x, (-1, input_dim))
    x = tf.matmul(x, w) + b
    x = tf.reshape(x, (-1, timesteps, output_dim))
    return x


class AttentionDecoder(Layer):
    """
    Bahdanau-style attention decoder (GRU variant).

    Implements the attention mechanism from:
        Bahdanau, Cho & Bengio, "Neural machine translation by jointly
        learning to align and translate", arXiv:1409.0473 (2014).

    Replaces the legacy keras.layers.recurrent.Recurrent subclass with a
    standard tf.keras.layers.Layer that manually unrolls timesteps.
    """

    def __init__(self, units, output_dim,
                 activation='tanh',
                 return_probabilities=False,
                 kernel_initializer='glorot_uniform',
                 recurrent_initializer='orthogonal',
                 bias_initializer='zeros',
                 kernel_regularizer=None,
                 bias_regularizer=None,
                 activity_regularizer=None,
                 kernel_constraint=None,
                 bias_constraint=None,
                 **kwargs):
        self.units = units
        self.output_dim = output_dim
        self.return_probabilities = return_probabilities
        self.activation = activations.get(activation)
        self.kernel_initializer = initializers.get(kernel_initializer)
        self.recurrent_initializer = initializers.get(recurrent_initializer)
        self.bias_initializer = initializers.get(bias_initializer)

        self.kernel_regularizer = regularizers.get(kernel_regularizer)
        self.recurrent_regularizer = regularizers.get(kernel_regularizer)
        self.bias_regularizer = regularizers.get(bias_regularizer)
        self.activity_regularizer_fn = regularizers.get(activity_regularizer)

        self.kernel_constraint = constraints.get(kernel_constraint)
        self.recurrent_constraint = constraints.get(kernel_constraint)
        self.bias_constraint = constraints.get(bias_constraint)

        super(AttentionDecoder, self).__init__(**kwargs)

    def build(self, input_shape):
        self.batch_size, self.timesteps, self.input_dim = input_shape

        # --- Attention weights ---
        self.V_a = self.add_weight(shape=(self.units,), name='V_a',
                                   initializer=self.kernel_initializer)
        self.W_a = self.add_weight(shape=(self.units, self.units), name='W_a',
                                   initializer=self.kernel_initializer)
        self.U_a = self.add_weight(shape=(self.input_dim, self.units), name='U_a',
                                   initializer=self.kernel_initializer)
        self.b_a = self.add_weight(shape=(self.units,), name='b_a',
                                   initializer=self.bias_initializer)

        # --- Reset gate ---
        self.C_r = self.add_weight(shape=(self.input_dim, self.units), name='C_r',
                                   initializer=self.recurrent_initializer)
        self.U_r = self.add_weight(shape=(self.units, self.units), name='U_r',
                                   initializer=self.recurrent_initializer)
        self.W_r = self.add_weight(shape=(self.output_dim, self.units), name='W_r',
                                   initializer=self.recurrent_initializer)
        self.b_r = self.add_weight(shape=(self.units,), name='b_r',
                                   initializer=self.bias_initializer)

        # --- Update gate ---
        self.C_z = self.add_weight(shape=(self.input_dim, self.units), name='C_z',
                                   initializer=self.recurrent_initializer)
        self.U_z = self.add_weight(shape=(self.units, self.units), name='U_z',
                                   initializer=self.recurrent_initializer)
        self.W_z = self.add_weight(shape=(self.output_dim, self.units), name='W_z',
                                   initializer=self.recurrent_initializer)
        self.b_z = self.add_weight(shape=(self.units,), name='b_z',
                                   initializer=self.bias_initializer)

        # --- Proposal ---
        self.C_p = self.add_weight(shape=(self.input_dim, self.units), name='C_p',
                                   initializer=self.recurrent_initializer)
        self.U_p = self.add_weight(shape=(self.units, self.units), name='U_p',
                                   initializer=self.recurrent_initializer)
        self.W_p = self.add_weight(shape=(self.output_dim, self.units), name='W_p',
                                   initializer=self.recurrent_initializer)
        self.b_p = self.add_weight(shape=(self.units,), name='b_p',
                                   initializer=self.bias_initializer)

        # --- Output projection ---
        self.C_o = self.add_weight(shape=(self.input_dim, self.output_dim), name='C_o',
                                   initializer=self.recurrent_initializer)
        self.U_o = self.add_weight(shape=(self.units, self.output_dim), name='U_o',
                                   initializer=self.recurrent_initializer)
        self.W_o = self.add_weight(shape=(self.output_dim, self.output_dim), name='W_o',
                                   initializer=self.recurrent_initializer)
        self.b_o = self.add_weight(shape=(self.output_dim,), name='b_o',
                                   initializer=self.bias_initializer)

        # --- Initial state ---
        self.W_s = self.add_weight(shape=(self.input_dim, self.units), name='W_s',
                                   initializer=self.recurrent_initializer)

        super(AttentionDecoder, self).build(input_shape)

    def call(self, x):
        # Pre-compute U_a * x + b_a for all timesteps
        uxpb = _time_distributed_dense(
            x, self.U_a, b=self.b_a,
            input_dim=self.input_dim,
            timesteps=self.timesteps,
            output_dim=self.units,
        )

        # Initial hidden state from first timestep
        s = activations.tanh(K.dot(x[:, 0], self.W_s))
        # Initial output token (zeros)
        y = tf.zeros((tf.shape(x)[0], self.output_dim))

        outputs = []
        for t in range(self.timesteps):
            _stm = K.repeat(s, self.timesteps)
            _Wxstm = K.dot(_stm, self.W_a)

            et = K.dot(activations.tanh(_Wxstm + uxpb),
                       K.expand_dims(self.V_a))
            at = K.exp(et)
            at_sum = K.sum(at, axis=1)
            at_sum_repeated = K.repeat(at_sum, self.timesteps)
            at = at / at_sum_repeated

            context = K.squeeze(K.batch_dot(at, x, axes=1), axis=1)

            rt = activations.sigmoid(
                K.dot(y, self.W_r) + K.dot(s, self.U_r)
                + K.dot(context, self.C_r) + self.b_r)

            zt = activations.sigmoid(
                K.dot(y, self.W_z) + K.dot(s, self.U_z)
                + K.dot(context, self.C_z) + self.b_z)

            s_prop = activations.tanh(
                K.dot(y, self.W_p) + K.dot(rt * s, self.U_p)
                + K.dot(context, self.C_p) + self.b_p)

            s = (1 - zt) * s + zt * s_prop

            y = activations.softmax(
                K.dot(y, self.W_o) + K.dot(s, self.U_o)
                + K.dot(context, self.C_o) + self.b_o)

            if self.return_probabilities:
                outputs.append(at)
            else:
                outputs.append(y)

        # Stack along time axis → (batch, timesteps, dim)
        return K.permute_dimensions(
            K.stack(outputs), (1, 0, 2)
        )

    def compute_output_shape(self, input_shape):
        if self.return_probabilities:
            return (input_shape[0], self.timesteps, self.timesteps)
        return (input_shape[0], self.timesteps, self.output_dim)

    def get_config(self):
        config = {
            'units': self.units,
            'output_dim': self.output_dim,
            'return_probabilities': self.return_probabilities,
        }
        base_config = super(AttentionDecoder, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))

In [0]:
# --- Helper functions ---

def generate_sequence(length, n_unique):
    return [randint(0, n_unique - 1) for _ in range(length)]


def one_hot_encode(sequence, n_unique):
    encoding = []
    for value in sequence:
        vector = [0] * n_unique
        vector[value] = 1
        encoding.append(vector)
    return array(encoding)


def one_hot_decode(encoded_seq):
    return [argmax(vector) for vector in encoded_seq]


def get_pair(n_in, n_out, cardinality):
    sequence_in = generate_sequence(n_in, cardinality)
    sequence_out = sequence_in[:n_out] + [0] * (n_in - n_out)
    X = one_hot_encode(sequence_in, cardinality)
    y = one_hot_encode(sequence_out, cardinality)
    X = X.reshape((1, X.shape[0], X.shape[1]))
    y = y.reshape((1, y.shape[0], y.shape[1]))
    return X, y


# --- Model builders ---

def baseline_model(n_timesteps_in, n_features):
    model = Sequential([
        LSTM(150, input_shape=(n_timesteps_in, n_features)),
        RepeatVector(n_timesteps_in),
        LSTM(150, return_sequences=True),
        TimeDistributed(Dense(n_features, activation='softmax')),
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model


def attention_model(n_timesteps_in, n_features):
    model = Sequential([
        LSTM(150, input_shape=(n_timesteps_in, n_features), return_sequences=True),
        AttentionDecoder(150, n_features),
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model


def train_evaluate_model(model, n_timesteps_in, n_timesteps_out, n_features):
    for _ in range(5000):
        X, y = get_pair(n_timesteps_in, n_timesteps_out, n_features)
        model.fit(X, y, epochs=1, verbose=0)
    total, correct = 100, 0
    for _ in range(total):
        X, y = get_pair(n_timesteps_in, n_timesteps_out, n_features)
        yhat = model.predict(X, verbose=0)
        if array_equal(one_hot_decode(y[0]), one_hot_decode(yhat[0])):
            correct += 1
    return float(correct) / float(total) * 100.0


# --- Run experiment ---

n_features = 50
n_timesteps_in = 5
n_timesteps_out = 2
n_repeats = 10

print('Encoder-Decoder Model')
results = []
for _ in range(n_repeats):
    model = baseline_model(n_timesteps_in, n_features)
    accuracy = train_evaluate_model(model, n_timesteps_in, n_timesteps_out, n_features)
    results.append(accuracy)
    print(accuracy)
print('Mean Accuracy: %.2f%%' % (sum(results) / float(n_repeats)))

print('Encoder-Decoder With Attention Model')
results = []
for _ in range(n_repeats):
    model = attention_model(n_timesteps_in, n_features)
    accuracy = train_evaluate_model(model, n_timesteps_in, n_timesteps_out, n_features)
    results.append(accuracy)
    print(accuracy)
print('Mean Accuracy: %.2f%%' % (sum(results) / float(n_repeats)))

In [0]:
# --- Vanilla LSTM regressor (template) ---
# Fill in n_units, X_train, Y_train, batch_size, and epochs
# before running this cell.

# n_units = 64
# regressor = Sequential([
#     LSTM(units=n_units, activation='sigmoid',
#          return_sequences=True, input_shape=(None, 1)),
#     LSTM(units=n_units, activation='sigmoid'),
#     Dense(units=1),
# ])
# regressor.compile(optimizer='adam', loss='mean_squared_error')
# regressor.fit(X_train, Y_train, batch_size=32, epochs=100)